# Stg helper — history_load (every read kept, grain = id × snapshot_ts)

Added 2026-11-02 (datle): Silver keeps **every read** of a source row instead of one row per day. Used with `%run ./_history` by the Silver notebooks of Bronze tables whose rows are read again over time (videos, channels, comments, places, search keywords, place search results).

**`history_load(source, target, key, ddl, ts_expr)`:**
1. **Migration (once):** a `target` created before `snapshot_ts` existed (old grain id × day, or id) is renamed to `<target>_bak_20261102` (kept, never deleted) and a new table is created. Its rows are copied once into the new table (`snapshot_ts` = their `ts_expr`), Bronze rows win on the same grain; the table property `history.bak_copied` records the copy.
2. **Incremental:** Bronze rows with `ts_expr >= MAX(snapshot_ts)` of the target; one row per grain (latest `ingest_ts`, Bronze can hold the same Postgres row twice).
3. **Insert-only MERGE** on (`key`, `snapshot_ts`): a rerun adds nothing twice, nothing is ever updated.
4. View **`<target>_latest`**: the latest read per `key` (same columns), for every task that needs the current value.
5. Grain check: fails on a duplicate (`key`, `snapshot_ts`).

`snapshot_ts` = when the source observed the values (`COALESCE(updated_at, created_at)` for Postgres rows the app upserts); `snapshot_date` = its Vietnam day (filter / clustering column). Aggregation per day / hour happens in the serving tables, not here.

In [ ]:
from pyspark.sql import Window
from pyspark.sql import functions as F

HISTORY_TZ = "Asia/Ho_Chi_Minh"
HISTORY_BAK_SUFFIX = "_bak_20261102"


def _table_props(table):
    return {r["key"]: r["value"] for r in spark.sql(f"SHOW TBLPROPERTIES {table}").collect()}


def history_load(source, target, key, ddl, ts_expr="COALESCE(updated_at, created_at)", transform=None,
                 latest_view=True, copy_bak=True):
    """Append the new reads of Bronze `source` to Silver `target` (grain key x snapshot_ts). Returns rows added.

    key: column name or list of names. ddl: column definitions of target, must include
    snapshot_ts TIMESTAMP and snapshot_date DATE. transform: optional function(DataFrame) -> DataFrame
    applied to the Bronze rows (after snapshot_ts is added) before they are conformed to the target columns.
    latest_view: create the generic <target>_latest view (False when the notebook defines its own).
    copy_bak: copy the rows of the renamed old table (False when Bronze holds all of them and the old
    table has other columns, e.g. stg_places).
    """
    keys = [key] if isinstance(key, str) else list(key)
    bak = f"{target}{HISTORY_BAK_SUFFIX}"
    if spark.catalog.tableExists(target) and "snapshot_ts" not in spark.table(target).columns:
        spark.sql(f"ALTER TABLE {target} RENAME TO {bak}")
        print(f"🔁 {target}: old grain, renamed to {bak} (kept) and rebuilt with snapshot_ts")
    spark.sql(f"""
        CREATE TABLE IF NOT EXISTS {target} ({ddl})
        USING DELTA
        CLUSTER BY (snapshot_date)
        TBLPROPERTIES ('delta.enableChangeDataFeed' = 'true')
    """)
    schema = spark.table(target).schema
    cols = [f.name for f in schema]
    types = {f.name: f.dataType.simpleString() for f in schema}

    def conform(df, priority):
        out = []
        for c in cols:
            if c == "snapshot_date":
                out.append(F.to_date(F.from_utc_timestamp(F.col("snapshot_ts"), HISTORY_TZ)).alias(c))
            elif c == "etl_date":
                out.append(F.current_timestamp().alias(c))
            elif c == "op_type":
                out.append(F.lit("INSERT").alias(c))
            elif c in df.columns:
                out.append(F.col(c).cast(types[c]).alias(c))
            else:
                out.append(F.lit(None).cast(types[c]).alias(c))
        return df.select(*out, F.lit(priority).alias("_priority"),
                         (F.col("ingest_ts") if "ingest_ts" in df.columns else F.lit(None).cast("timestamp")).alias("_ingest_ts"))

    parts = []
    wm = spark.sql(f"SELECT MAX(snapshot_ts) AS wm FROM {target}").collect()[0]["wm"]
    if spark.catalog.tableExists(source):
        df = spark.table(source).withColumn("snapshot_ts", F.expr(ts_expr))
        if wm is not None:
            df = df.filter(F.col("snapshot_ts") >= F.lit(wm))  # >=: rows sharing the last ts; the MERGE skips the ones already in
            print(f"💧 {target}: watermark {wm}")
        else:
            print(f"📦 {target}: full load from {source}")
        if transform is not None:
            df = transform(df)
        parts.append(conform(df, 0))
    else:
        print(f"⏭️ {source} does not exist yet")
    copy_bak = copy_bak and spark.catalog.tableExists(bak) and _table_props(target).get("history.bak_copied") != "true"
    if copy_bak:
        bdf = spark.table(bak).withColumn("snapshot_ts", F.expr(ts_expr))
        if transform is not None:
            bdf = transform(bdf)
        parts.append(conform(bdf, 1))
        print(f"📥 {target}: copying the rows of {bak} once")
    added = 0
    if parts:
        src = parts[0]
        for p in parts[1:]:
            src = src.unionByName(p)
        one = Window.partitionBy(*keys, "snapshot_ts").orderBy(F.col("_priority"), F.col("_ingest_ts").desc_nulls_last())
        src = (src.where(" AND ".join(f"{k} IS NOT NULL" for k in keys + ["snapshot_ts"]))
                  .withColumn("_rn", F.row_number().over(one)).where("_rn = 1")
                  .drop("_rn", "_priority", "_ingest_ts"))
        src.createOrReplaceTempView("history_src")
        before = spark.table(target).count()
        on = " AND ".join(f"t.{k} = s.{k}" for k in keys + ["snapshot_ts"])
        spark.sql(f"MERGE INTO {target} AS t USING history_src AS s ON {on} WHEN NOT MATCHED THEN INSERT *")
        if copy_bak:
            spark.sql(f"ALTER TABLE {target} SET TBLPROPERTIES ('history.bak_copied' = 'true')")
        added = spark.table(target).count() - before

    if latest_view:
        spark.sql(f"""
        CREATE OR REPLACE VIEW {target}_latest
        COMMENT 'Latest read per {", ".join(keys)} of {target} (same columns).'
        AS SELECT * FROM {target}
        QUALIFY ROW_NUMBER() OVER (PARTITION BY {", ".join(keys)} ORDER BY snapshot_ts DESC) = 1
    """)
    g = spark.sql(f"""SELECT COUNT(*) AS n, COUNT(DISTINCT {", ".join(keys)}, snapshot_ts) AS grain,
                             COUNT(DISTINCT {", ".join(keys)}) AS ids FROM {target}""").collect()[0]
    print(f"✅ {target}: +{added} reads · {g['n']} reads of {g['ids']} ids · grain ({', '.join(keys)}, snapshot_ts): "
          + ("✅" if g["n"] == g["grain"] else "❌ duplicates"))
    if g["n"] != g["grain"]:
        raise AssertionError(f"{target}: {g['n'] - g['grain']} duplicate ({', '.join(keys)}, snapshot_ts) rows")
    return added